# AutoGen Demo 1: Multi-Agent Color Chain Game

In this demo, we introduce one of the core ideas behind agentic AI systems:  
multiple AI agents communicating through a shared conversation and responding to each other over time.

Unlike a traditional single-prompt workflow, the behavior here emerges from interaction:
- one agent makes a move
- another agent must interpret that move and respond correctly
- a separate judge validates the interaction and enforces rules
- shared conversational state determines what happens next

This is a small example, but it demonstrates an important shift in architecture.  
The "computation" is no longer just a single LLM response. The conversation itself becomes part of the system logic.

---

## Why AutoGen Is Interesting

Frameworks like AutoGen focus on conversation-oriented multi-agent systems.

Instead of building a rigid workflow graph first, AutoGen treats agents as participants in an ongoing interaction:
- agents can specialize into roles
- agents can critique or refine each other
- agents can collaborate or compete
- transcripts become observable and debuggable
- behavior emerges from message exchange over time

This differs from frameworks like LangGraph, where the primary abstraction is usually workflow state and routing logic.

A useful rule of thumb:

- **AutoGen** is strongest when the *interaction between agents* is the interesting part.
- **LangGraph** is strongest when the *workflow and state transitions* are the interesting part.

Neither approach is universally better. They optimize for different kinds of systems.

---

## Game Rules

1. The first move can be any valid color.
2. Every later color must start with the last letter of the previous color.
3. A color cannot be reused.
4. Spaces and hyphens are treated equivalently (`forest-green` == `forest green`).
5. The color must be short enough to avoid nonsense like `greenish-blueish-greyish`.
6. First invalid move loses.

---

## What This Demo Teaches

This demo intentionally focuses on constrained interaction rather than deep reasoning.

The agents are not "thinking" strategically. Instead, they are:
- reading conversational context
- maintaining short-term memory
- following shared rules
- reacting to prior messages
- succeeding or failing based on context quality

This makes the demo useful for exploring:
- conversational state
- context windows
- structured validation
- multi-agent coordination
- failure modes caused by missing context

We will also experiment with different context window sizes to observe how reduced conversational memory impacts agent performance.

---

## Important Practical Note

AutoGen is excellent for demonstrating conversational multi-agent patterns and rapid experimentation.

However, modern production systems often combine these ideas with more structured orchestration frameworks such as LangGraph, especially when:
- deterministic workflows matter
- retries and checkpoints are required
- state management becomes complex
- human approvals or tool routing must be tightly controlled

Additionally, the original Microsoft AutoGen project is now in maintenance mode.  
The concepts remain valuable, but the ecosystem is evolving quickly, and production architectures increasingly mix conversational-agent ideas with graph- and workflow-oriented systems.

## Setup

This notebook uses the modern AutoGen AgentChat packages.

Run the install cell if needed. Then set `OPENAI_API_KEY` in your environment or in a local `.env` file.

```bash
pip install -U autogen-agentchat autogen-ext[openai] python-dotenv pandas
```

AutoGen is currently best treated as a teaching/prototyping framework here, not as a production commitment. The important lesson is the multi-agent communication pattern.

In [1]:
# Optional install cell. Uncomment if your environment does not already have these packages.
#!pip install -U autogen-agentchat "autogen-ext[openai]" python-dotenv pandas

In [ ]:
# Install color list and save into colors.txt
import pandas as pd
import re
LOAD_COLORS = False

if LOAD_COLORS:
    url = "https://raw.githubusercontent.com/codebrainz/color-names/master/output/colors.csv"

    df = pd.read_csv(url, sep=",", quotechar='"',names=['default','color','hex','R','G','B'])
    colors = (
        df['color']
        .str.lower()
        .str.strip()
        .drop_duplicates()
        .sort_values()
    )
    colors = [re.sub(r"\s*\([^)]*\)", "", color) for color in colors.values]
    colors = list(set(colors))
    colors = pd.DataFrame(data=colors, columns=['color'])

    colors.to_csv("colors.txt", index=False, header=False)

    print(f"Wrote {len(colors)} colors")

In [3]:
import asyncio
import os
import re
from dataclasses import dataclass, field
from typing import Literal

import pandas as pd
from dotenv import load_dotenv

load_dotenv()

MODEL = os.getenv("OPENAI_MODEL", "gpt-4.1-mini")
TEMPERATURE = float(os.getenv("OPENAI_TEMPERATURE", "0.2"))

print(f"Using model: {MODEL}")

Using model: gpt-4.1-mini


## Game configuration

The `context_window_turns` setting is the main teaching dial.

Try:
- `1`: agents see only the last move and recent used colors may be hidden.
- `5`: agents usually do fine.
- `100`: agents see nearly everything.

The `max_color_length` setting prevents cheap outputs like `greenish-blue-gray-purple`.

In [4]:
@dataclass
class GameConfig:
    max_turns: int = 12
    context_window_turns: int = 5
    max_color_length: int = 12
    starting_color: str | None = "white"
    strict_css_colors_only: bool = False

config = GameConfig(
    max_turns=12,
    context_window_turns=5,
    max_color_length=12,
    starting_color="white",
    strict_css_colors_only=False,
)

config

GameConfig(max_turns=12, context_window_turns=5, max_color_length=12, starting_color='white', strict_css_colors_only=False)

## Deterministic validation

The LLMs play the game, but ordinary Python enforces the obvious rules:
- first-letter / last-letter match
- duplicate detection
- length limit
- single color token format

The only fuzzy part is whether a word is plausibly a color. For class stability, this demo uses a curated color list plus common non-CSS color names such as `taupe`, `ecru`, and `eggplant`.

In [5]:
from pathlib import Path
import re

def normalize_color(raw: str) -> str:
    text = raw.strip().lower()
    text = text.strip('"').strip("'")
    text = re.sub(r"\s*\([^)]*\)", "", text)
    text = re.sub(r"[.!?;,:\s]+$", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def color_key(color: str) -> str:
    """
    Canonical key for lookup and duplicate detection.

    Treat these as equivalent:
    - navy blue
    - navy-blue
    - navyblue
    """
    return re.sub(r"[\s\-]+", "", normalize_color(color))


def load_color_dictionary(path="colors.txt"):
    colors = {}

    for line in Path(path).read_text(encoding="utf-8").splitlines():
        color = normalize_color(line)
        if not color:
            continue

        key = color_key(color)
        colors[key] = color

    return colors

VALID_COLORS = load_color_dictionary("colors.txt")

@dataclass
class MoveResult:
    valid: bool
    reason: str
    normalized_color: str


def validate_move(raw_candidate, previous_color, used_color_keys, valid_colors, max_len=20):
    candidate = normalize_color(raw_candidate)
    candidate_key = color_key(candidate)

    if not candidate:
        return False, candidate, "No color was provided."

    if len(candidate) > max_len:
        return False, candidate, f"Color is too long. Max length is {max_len} characters."

    if candidate_key not in valid_colors:
        return False, candidate, "Color is not in the approved color dictionary."

    if candidate_key in used_color_keys:
        return False, candidate, "Color was already used."

    if previous_color:
        expected = color_key(previous_color)[-1]
        actual = candidate_key[0]

        if actual != expected:
            return False, candidate, f"Color must start with '{expected}', but starts with '{actual}'."

    canonical_color = valid_colors[candidate_key]
    return True, canonical_color, "Valid move."

## AutoGen agents

The two players are intentionally given slightly different strategies. That makes the transcript more interesting, but both still follow the same rules.

In [6]:
from autogen_agentchat.agents import AssistantAgent
from autogen_ext.models.openai import OpenAIChatCompletionClient


def make_model_client():
    # Uses OPENAI_API_KEY from your environment.
    return OpenAIChatCompletionClient(
        model=MODEL,
        temperature=TEMPERATURE,
    )


def make_color_agent(name: str, style: Literal["safe", "creative"]):
    if style == "safe":
        strategy = "Prefer common, short color names. Avoid obscure words unless necessary."
    else:
        strategy = "Prefer valid but less obvious color names when possible. Still obey every rule."

    system_message = f"""
You are {name}, a player in the Color Chain Game.

Rules:
- Reply with exactly one color name and nothing else.
- Use only letters. No punctuation. No explanations.
- The color must start with the required letter.
- The color must not appear in the used color list.
- The color must be no longer than the configured max length.
- If you are uncertain, choose a simple common color.

Strategy:
{strategy}
""".strip()

    return AssistantAgent(
        name=name,
        model_client=make_model_client(),
        system_message=system_message,
    )

safe_agent = make_color_agent("SafeColorAgent", "safe")
creative_agent = make_color_agent("CreativeColorAgent", "creative")

safe_agent.name, creative_agent.name

('SafeColorAgent', 'CreativeColorAgent')

## Game state and prompt construction

We keep the full game state in Python, but each agent only receives a configurable slice of recent history.

That lets students see the difference between:
- **system state**: what the program knows
- **agent context**: what the LLM sees this turn

In [7]:
@dataclass
class GameMove:
    turn: int
    agent: str
    raw_response: str
    color: str
    valid: bool
    judge_reason: str


@dataclass
class GameState:
    previous_color: str | None = None
    used_color_keys: set[str] = field(default_factory=set)
    moves: list[GameMove] = field(default_factory=list)
    finished: bool = False
    winner: str | None = None
    loser: str | None = None


def build_turn_prompt(state: GameState, config: GameConfig) -> str:
    previous = state.previous_color or "none"
    required_letter = "any" if state.previous_color is None else state.previous_color[-1]

    visible_moves = state.moves[-config.context_window_turns:]
    visible_history = [m.color for m in visible_moves]

    # Teaching twist: with a small context window, hide older used colors from the player.
    # The judge still knows the full state, so the player can accidentally repeat an older color.
    visible_used = visible_history.copy()

    return f"""
Current game state visible to you:
- Previous color: {previous}
- Required starting letter: {required_letter}
- Recently visible used colors: {visible_used}
- Max color length: {config.max_color_length}

Your task:
Reply with exactly one valid color name.
No explanation. No punctuation. One color only.
""".strip()

## Run the game

The loop below is intentionally explicit.

AutoGen produces each player's move. Python validates the move. The transcript makes the communication pattern visible.

In [8]:
async def ask_agent_for_color(agent: AssistantAgent, state: GameState, config: GameConfig) -> str:
    prompt = build_turn_prompt(state, config)
    result = await agent.run(task=prompt)
    return str(result.messages[-1].content).strip()

async def run_color_game(config: GameConfig):
    state = GameState()

    if config.starting_color:
        first = normalize_color(config.starting_color)

        is_valid_start, first_candidate, start_reason = validate_move(
            raw_candidate=first,
            previous_color=None,
            used_color_keys=state.used_color_keys,
            valid_colors=VALID_COLORS,
            max_len=config.max_color_length,
        )

        if not is_valid_start:
            raise ValueError(f"Invalid starting color '{config.starting_color}': {start_reason}")

        state.previous_color = first_candidate
        state.used_color_keys.add(first_candidate)
        state.moves.append(
            GameMove(
                turn=0,
                agent="Starter",
                raw_response=config.starting_color,
                color=first_candidate,
                valid=True,
                judge_reason="Starting color.",
            )
        )

    agents = [safe_agent, creative_agent]

    print("COLOR CHAIN GAME")
    print(f"Context window: {config.context_window_turns} turns")
    print(f"Max color length: {config.max_color_length}")
    print("-" * 60)

    if config.starting_color:
        print(f"[Starter]: {state.previous_color}")

    for turn in range(1, config.max_turns + 1):
        agent = agents[(turn - 1) % len(agents)]

        raw = await ask_agent_for_color(agent, state, config)

        is_valid, candidate, reason = validate_move(
            raw_candidate=raw,
            previous_color=state.previous_color,
            used_color_keys=state.used_color_keys,
            valid_colors=VALID_COLORS,
            max_len=config.max_color_length,
        )

        move = GameMove(
            turn=turn,
            agent=agent.name,
            raw_response=raw,
            color=candidate,
            valid=is_valid,
            judge_reason=reason,
        )
        state.moves.append(move)

        print(f"[{agent.name}]: {raw}")
        print(f"[Judge]: {reason}")

        if not is_valid:
            state.finished = True
            state.loser = agent.name
            state.winner = agents[turn % len(agents)].name

            print("-" * 60)
            print(f"Winner: {state.winner}")
            print(f"Loser: {state.loser}")
            break

        state.previous_color = candidate
        state.used_color_keys.add(color_key(candidate))

    if not state.finished:
        print("-" * 60)
        print("No invalid moves. The game ended by turn limit.")

    return state

state = await run_color_game(config)

COLOR CHAIN GAME
Context window: 5 turns
Max color length: 12
------------------------------------------------------------
[Starter]: white
[SafeColorAgent]: emerald
[Judge]: Valid move.
[CreativeColorAgent]: denim
[Judge]: Valid move.
[SafeColorAgent]: magenta
[Judge]: Valid move.
[CreativeColorAgent]: amber
[Judge]: Valid move.
[SafeColorAgent]: red
[Judge]: Valid move.
[CreativeColorAgent]: dandelion
[Judge]: Valid move.
[SafeColorAgent]: navy
[Judge]: Color is not in the approved color dictionary.
------------------------------------------------------------
Winner: CreativeColorAgent
Loser: SafeColorAgent


## Review the transcript as a table

This makes it easier to discuss failure modes after the live run.

In [9]:
df = pd.DataFrame([m.__dict__ for m in state.moves])
df

,turn,agent,raw_response,color,valid,judge_reason
0,0,Starter,white,white,True,Starting color.
1,1,SafeColorAgent,emerald,emerald,True,Valid move.
2,2,CreativeColorAgent,denim,denim,True,Valid move.
3,3,SafeColorAgent,magenta,magenta,True,Valid move.
4,4,CreativeColorAgent,amber,amber,True,Valid move.
5,5,SafeColorAgent,red,red,True,Valid move.
6,6,CreativeColorAgent,dandelion,dandelion,True,Valid move.
7,7,SafeColorAgent,navy,navy,False,Color is not in the approved color dictionary.


## Experiment: change the context window

Run the same game with different values.

Suggested tests:
- `context_window_turns=1`: likely to repeat older colors eventually.
- `context_window_turns=5`: usually workable.
- `context_window_turns=100`: best memory, but more prompt context.

This is a good place to ask students:

> What should be stored in application state vs. passed through the LLM context window?

In [14]:
config_small_context = GameConfig(
    max_turns=25,
    context_window_turns=10,
    max_color_length=12,
    starting_color="white",
    strict_css_colors_only=False,
)

state_small_context = await run_color_game(config_small_context)

COLOR CHAIN GAME
Context window: 10 turns
Max color length: 12
------------------------------------------------------------
[Starter]: white
[SafeColorAgent]: eggplant
[Judge]: Valid move.
[CreativeColorAgent]: teal
[Judge]: Valid move.
[SafeColorAgent]: lavender
[Judge]: Valid move.
[CreativeColorAgent]: ruby
[Judge]: Valid move.
[SafeColorAgent]: yellow
[Judge]: Valid move.
[CreativeColorAgent]: wheat
[Judge]: Valid move.
[SafeColorAgent]: turquoise
[Judge]: Valid move.
[CreativeColorAgent]: eggshell
[Judge]: Valid move.
[SafeColorAgent]: lime
[Judge]: Valid move.
[CreativeColorAgent]: ecru
[Judge]: Valid move.
[SafeColorAgent]: umber
[Judge]: Valid move.
[CreativeColorAgent]: rose
[Judge]: Valid move.
[SafeColorAgent]: emerald
[Judge]: Valid move.
[CreativeColorAgent]: denim
[Judge]: Valid move.
[SafeColorAgent]: magenta
[Judge]: Valid move.
[CreativeColorAgent]: apricot
[Judge]: Valid move.
[SafeColorAgent]: teal
[Judge]: Color was already used.
------------------------------------

## Teaching notes

This demo is intentionally small, but it introduces several serious agent concepts.

### What AutoGen adds here

AutoGen gives us named agents with persistent role instructions and a conversation-oriented programming model. The official AgentChat docs describe agents as named participants with a `run` method that returns a message history, and `AssistantAgent` as a model-backed agent that can also use tools.

### What Python adds here

Python owns the rules. That is not a shortcut. It is good system design.

Use deterministic code for things code can check:
- duplicate detection
- max length
- first-letter / last-letter match
- stop conditions

Use LLMs for the part where language flexibility matters:
- choosing a plausible color
- adapting to visible context
- following conversational constraints

### What students should notice

1. More context helps, but costs more tokens.
2. Less context can cause repeated colors or illegal moves.
3. A judge or validator is essential when agents can fail.
4. More agents does not automatically mean better behavior.
5. Good agent design is usually a mix of LLM behavior and conventional software guardrails.

### Suggested transition to Demo 2

This game shows constrained interaction.

Next, we can increase the stakes by giving agents conflicting goals and asking them to debate a semi-serious engineering question.

## Optional extension ideas

- Add a real `JudgeAgent` that only decides whether obscure words are valid color names.
- Add a scoreboard across multiple rounds.
- Add a `HintAgent` that suggests legal next letters when players get stuck.
- Compare this explicit loop to `RoundRobinGroupChat`.
- Replace colors with programming languages, cities, animals, or chemical elements.

For teaching, I would keep the first pass simple. The point is not the game. The point is communication, shared state, and validation.